# CHAT Feature Engineering Pipeline
This notebook implements the CHATFeatureEngineer class, a robust automated pipeline designed to transform raw CHAT (.cha) transcripts from the CHILDES database into structured, analysis-ready Parquet files.

## Key Features:
* Exhaustive Metadata Extraction: Captures session-level details across 21 standard CHAT headers (e.g., date, location, situation) and speaker-level metadata (e.g., age in months, role).
  
* Linguistic Metrics: Automatically computes standardized developmental measures using PyLangAcq, including MLU (Mean Length of Utterance in words and morphemes), TTR (Type-Token Ratio), and IPSyn (Index of Productive Syntax).
  
* Granular NLP Tiers: Uses regex-based parsing to extract and aggregate morphological information (%mor), grammatical relations (%gra), and phonological data (%xpho) for every utterance in a session.
  
* Conversational Context: Tracks the immediate temporal context by capturing the preceding and following utterances for every speaker turn.
  
* Hierarchical Processing: Recursively mirrors the raw data directory structure, exporting individual experiment files and generating a concatenated meta-file (e.g., _Eng-NA_extracted.parquet) for each language/study group.
  
* Data Integrity: Implements sophisticated logging and safety checks (_get_tier_safe, try-except blocks) to ensure high-throughput processing even when encountering inconsistent or legacy transcript formats.

In [49]:
import re
import pandas as pd
import numpy as np
import pylangacq
from pathlib import Path
from typing import List, Dict, Any
import logging

# Setup sophisticated logging for high-throughput processing
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)
logger = logging.getLogger(__name__)

# Create class to process data automatically
class CHATFeatureEngineer:
    """
    An expert-level engineering pipeline for transforming CHAT transcripts into
    structured speaker-session level Parquet files with exhaustive metadata.
    """
    
    def __init__(self, data_root: str):
        self.data_root = Path(data_root)

        self.target_headers = [
            'comments', 'date', 'languages', 'location', 'media', 
            'number', 'options', 'other', 'participants', 'pid', 
            'recording_quality', 'room_layout', 'situation', 
            'tape_location', 'time_duration', 'time_start', 
            'transcriber', 'transcription', 'types', 'videos', 'warning'
        ] # All 21 headers specified in the CHAT standard and user query
        
    def _extract_provenance(self, file_path: Path) -> Dict[str, str]:
        """
        Parses the PosixPath to extract experimental hierarchy.
        Target Format:../data/raw/Eng-NA/Brown/Eve/010600b.cha
        """
        parts = file_path.parts
        # We walk backwards from the filename to ensure robustness
        # filename = parts[-1], subject = parts[-2], experiment = parts[-3], lang = parts[-4]
        try:
            return {
                'language_id': parts[-4],
                'experiment_name': parts[-3],
                'subject_id': parts[-2],
                'session_filename': file_path.name,
                'session_id': file_path.stem
            }
        except IndexError:
            logger.error(f"Path structure invalid for {file_path}. Ensure folder hierarchy is maintained.")
            return {k: 'unknown' for k in ['language_id', 'experiment_name', 'subject_id', 'session_filename', 'session_id']}

    def _calculate_conversational_features(self, utterances: List[Any], speaker_code: str) -> List:
        """
        Extracts temporal and contextual features at the utterance level for aggregation.
        """
        features = []
        
        for i, utt in enumerate(utterances):

            # 1. Handle Timestamps: time_marks is usually a tuple (start, end)
            time_marks = getattr(utt, 'time_marks', None)
            
            start = np.nan
            end = np.nan
            duration = np.nan
            
            if time_marks and len(time_marks) == 2:
                start = time_marks[0]
                end = time_marks[1]
                duration = end - start

            # 2. Handle Text Extraction
            def get_text(u):
                if u and hasattr(u, 'tokens') and u.tokens:
                    return " ".join([t.word for t in u.tokens if hasattr(t, 'word')])
                return None

            # 3. Contextual capturing (n-1 and n+1) from the full list provided
            prev_text = get_text(utterances[i-1]) if i > 0 else None
            next_text = get_text(utterances[i+1]) if i < len(utterances) - 1 else None
            
            features.append({
                'speaker': utt.participant,  # Vital for filtering later
                'start_time': start,
                'end_time': end,
                'duration': duration,
                'prev_context': prev_text,
                'next_context': next_text
            })
            
        return features

    def _get_tier_safe(self, utt_obj, tier_name):
        """
        Extracts tiers from CHAT files for metadata and feature extraction
        """
        if utt_obj and hasattr(utt_obj, 'tiers'): # tiers is a dictionary mapping tier IDs to the raw string

            val = utt_obj.tiers.get(tier_name, np.nan)

            return val if val != "" else np.nan
        
        return np.nan

    def process_corpus(self, output_base: str):
        """
        Main execution loop. Recursively identifies experiments and processes speakers.
        """

        # Defining paths, regex patterns
        output_path = Path(output_base)
        pos_regex = r'(\w+)(?=\|)'
        mor_regex = r'(?<=\|)([^\s]+)'
        gra_rel_regex = r'\|([^|\s]+)(?=\s|$)'
        gra_dep_regex = r'(\d+\|\d+)(?=\|)'

        language_aggregates = {}
        
        # Recursively find directories containing .cha files using rglob (this ensures we find '[language]/[study]' regardless of depth)
        processed_dirs = set()
        for cha_file in self.data_root.rglob("*.cha"):
            experiment_dir = cha_file.parent
            if experiment_dir in processed_dirs:
                continue
            processed_dirs.add(experiment_dir)
            
            # Calculating relative path to define output path (e.g., if data_root is 'raw' and experiment_dir is 'raw/Eng-NA/Brown' relative_path becomes 'Eng-NA')
            relative_path = experiment_dir.parent.relative_to(self.data_root)
            lang_out_dir = output_path / relative_path
            lang_out_dir.mkdir(parents=True, exist_ok=True)
            
            # Experiment Logger
            logger.info(f"Ingesting Data: {experiment_dir.name}")
            session_data = []
            
            # Load experiment via PyLangAcq Reader
            try:
                reader = pylangacq.read_chat(str(experiment_dir))
            except Exception as e:
                logger.error(f"PyLangAcq failed to read directory {experiment_dir}: {e}")
                continue

            # Process file-by-file to maintain metadata alignment 
            file_paths = reader.file_paths
            all_headers = reader.headers()
            
            for idx, f_path in enumerate(file_paths):
                p_path = Path(f_path)
                provenance = self._extract_provenance(p_path)
                file_header = all_headers[idx]
                                    
                # Identify all unique participants in this specific session 
                participants_object = reader.participants(by_file=True)[idx]

                # Capture roles for all participants
                code_to_role = {}

                for p in participants_object:
                    code_to_role[p.code] = p.role if p.role else np.nan 
                
                # Unique list of codes to loop through
                participants = list(code_to_role.keys())
                
                
                for speaker in participants:

                    try:
                        # 1. Get the FULL conversation for this file (to see other people)
                        full_session_reader = reader.filter(files=f_path)
                        all_session_utts = full_session_reader.utterances()

                        # 2. Isolate just this speaker for their specific metrics
                        speaker_reader = full_session_reader.filter(participants=[speaker])
                        speaker_utts = speaker_reader.utterances()
                        
                        # 3. Derived metrics extraction -- mean length of utterance (raw, words, morphemes), type-token ratio and Index of Productive Syntax (IPSyn)
                        mlu = speaker_reader.mlu(participant=speaker)[0] if speaker_reader.mlu(participant=speaker) else np.nan
                        mluw = speaker_reader.mluw(participant=speaker)[0] if speaker_reader.mluw(participant=speaker) else np.nan
                        mlum = speaker_reader.mlum(participant=speaker)[0] if speaker_reader.mlum(participant=speaker) else np.nan
                        ttr = speaker_reader.ttr(participant=speaker)[0] if speaker_reader.ttr(participant=speaker) else np.nan
                        
                        try: # for child
                            ipsyn = speaker_reader.ipsyn(participant=speaker)[0] if speaker_reader.ipsyn(participant=speaker) else np.nan
                        except: # for non-child
                            ipsyn = np.nan


                        # 4. Get age and speaker role metadata
                        ages_list = speaker_reader.ages()
                        age = ages_list[0] if ages_list else np.nan
                        age_months = age.years * 12 + age.months + age.days / 30 if age else np.nan

                        speaker_role = code_to_role.get(speaker, np.nan)

                        
                        # 5. Extract raw text and conversational contextual features -- raw text, morphological information, grammatical relations, and part of speech
                        raw_text_list = []
                        mor_info_full_list = []
                        gra_info_full_list = []
                        xpho_info_list = []
                        pos_info_list = []
                        mor_info_list = []
                        gra_rel_list = []
                        gra_dep_list = []

                        for utt in speaker_utts:

                            # Raw data extraction from utterances
                            u_raw = self._get_tier_safe(utt, speaker)
                            u_mor_full = self._get_tier_safe(utt, '%mor')
                            u_gra_full = self._get_tier_safe(utt, '%gra')
                            u_xpho = self._get_tier_safe(utt, '%xpho') # xpho - phonological information (word sound a.k.a. how it was said)

                            # Regex-based sub-extraction (Handling np.nan cases)
                            u_pos = " ".join(re.findall(pos_regex, str(u_mor_full))) if pd.notna(u_mor_full) else np.nan # pos - part of speech (noun, pronoun, verb, adjective, adverb, preposition, conjunction, and interjection)
                            u_mor = " ".join(re.findall(mor_regex, str(u_mor_full))) if pd.notna(u_mor_full) else np.nan # mor - morphological information (root word + form + modificaitons)
                            u_gra_rel = " ".join(re.findall(gra_rel_regex, str(u_gra_full))) if pd.notna(u_gra_full) else np.nan # rel - gramatical relation type
                            u_gra_dep = " ".join(re.findall(gra_dep_regex, str(u_gra_full))) if pd.notna(u_gra_full) else np.nan # dep | rel - dep = position of dependent word, head = position of head word
                            
                            # Append utterance-level data to speaker-level lists
                            raw_text_list.append(u_raw)
                            mor_info_full_list.append(u_mor_full)
                            gra_info_full_list.append(u_gra_full)
                            xpho_info_list.append(u_xpho)
                            pos_info_list.append(u_pos)
                            mor_info_list.append(u_mor)
                            gra_rel_list.append(u_gra_rel)
                            gra_dep_list.append(u_gra_dep)

                        # 6. Contextual features calculation -- previous and following utterances
                        temporal_context = self._calculate_conversational_features(all_session_utts, speaker)
                        speaker_features = [f for f in temporal_context if f['speaker'] == speaker]
                        prev_contexts = [f['prev_context'] for f in speaker_features]
                        next_contexts = [f['next_context'] for f in speaker_features] 
                                                    
                        
                        # Build the complete row dictionary
                        row = {
                            # Session Level Metadata
                            **provenance,
                            'speaker_code': speaker, # unique code (may refer to name)
                            'speaker_role': speaker_role, # classification of role relative to child
                            'speaker_age_months': age_months, # age of speaker (usually only child) in months
                            'mlu_score': mlu, # mean length of utterance raw
                            'mluw_score': mluw, # mean length of utterance in words
                            'mlum_score': mlum, # mean length of utterance in morphemes
                            'ttr_score': ttr, # type token ratio for non-punctuation words
                            'ipsyn_score': ipsyn, # index of productive syntax -- tool for assessing children's language development by scoring 59–60 specific grammatical forms within a 100-utterance sample
                            'n_utterances': len(speaker_utts),
                            
                            # Aggregated Utterance Lists (Stored as strings to maintain Parquet compatibility)
                            'text_part_of_speech': str(pos_info_list), # part of speech (noun, pronoun, verb, adjective, adverb, preposition, conjunction, and interjection)
                            'text_morphological_info': str(mor_info_list), # morphological information (root word + form + modificaitons)
                            'text_grammatical_relation_type': str(gra_rel_list), # gramatical relation type
                            'text_grammatical_dependency': str(gra_dep_list), # dep | rel - dep = position of dependent word, head = position of head word
                            'text_phonological_info': str(xpho_info_list), # phonological information (word sound aka how it was said)
                            'text_raw': str(raw_text_list), # verbatim text from the speaker tier across all utterances
                            
                            # Full tier strings for potential re-parsing
                            'text_morphological_info_full': str(mor_info_full_list), # the complete unparsed %mor tier for every utterance
                            'text_grammatical_relation_full': str(gra_info_full_list), # the complete unparsed %gra tier for every utterance
                            
                            # Conversational Context Lists
                            'context_previous_utterance': str(prev_contexts), # text of the utterance immediately preceding each speaker turn
                            'context_next_utterance': str(next_contexts) # text of the utterance immediately following each speaker turn
                        }

                        
                        for h_key in self.target_headers:

                            # Dynamically access the attributes for CHAT files (e.g., file_header.comments, file_header.date)
                            val = getattr(file_header, h_key, np.nan)
                            
                            # Clean the data for Parquet compatibility (Complex objects (lists/dicts) should be stringified; empty strings to NaN)
                            if isinstance(val, (list, dict, set)):
                                row[f"meta_{h_key}"] = str(val) if val else np.nan
                            elif val == "" or val is None:
                                row[f"meta_{h_key}"] = np.nan
                            else:
                                row[f"meta_{h_key}"] = val

                        session_data.append(row)
                        
                    except Exception as inner_e:
                        logger.warning(f"Failed processing speaker {speaker} in {provenance['session_id']}: {inner_e}")

            # Save individual data file
            if session_data:

                # Convert to DataFrame
                df = pd.DataFrame(session_data)
                df.replace(['', 'None', 'nan', 'NaN', '', '{}'], np.nan, inplace=True)
                
                # Export to Parquet
                output_file = lang_out_dir / f"{experiment_dir.name}_extracted.parquet"
                df.to_parquet(output_file)
                logger.info(f"Successfully exported {len(df)} records to {output_file}")

                # TRACK FOR METAFILE: Add df to its language group
                if relative_path not in language_aggregates:
                    language_aggregates[relative_path] = []
                language_aggregates[relative_path].append(df)

        # --- AFTER ALL LOOPS: CREATE METAFILES ---
        for lang_rel_path, df_list in language_aggregates.items():

            if df_list:

                # Concatenate all experiments for this language
                meta_df = pd.concat(df_list, ignore_index=True)
                
                # The filename: prefix with underscore and save in the parent folder
                meta_filename = f"_{lang_rel_path.name}_extracted.parquet"
                meta_output_path = output_path / lang_rel_path / meta_filename
                
                meta_df.to_parquet(meta_output_path)
                logger.info(f"Created METAFILE: {meta_output_path} with {len(meta_df)} total records.")

In [50]:
# Run CHAT data engineer class
engineer = CHATFeatureEngineer(data_root="../data/raw")
engineer.process_corpus(output_base="../data/processed")

2026-03-15 06:14:59,004 - [INFO] - Ingesting Data: Eve
2026-03-15 06:15:00,726 - [INFO] - Successfully exported 93 records to ../data/processed/Eng-NA/Brown/Eve_extracted.parquet
2026-03-15 06:15:00,726 - [INFO] - Ingesting Data: Adam
2026-03-15 06:15:05,453 - [INFO] - Successfully exported 248 records to ../data/processed/Eng-NA/Brown/Adam_extracted.parquet
2026-03-15 06:15:05,454 - [INFO] - Ingesting Data: Sarah
2026-03-15 06:15:10,247 - [INFO] - Successfully exported 557 records to ../data/processed/Eng-NA/Brown/Sarah_extracted.parquet
2026-03-15 06:15:10,310 - [INFO] - Created METAFILE: ../data/processed/Eng-NA/Brown/_Brown_extracted.parquet with 898 total records.


In [46]:
df = pd.read_parquet('../data/processed/Eng-NA/Brown/_Brown_extracted.parquet')
display(df.info(), df)

<class 'pandas.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 45 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   language_id                     898 non-null    str    
 1   experiment_name                 898 non-null    str    
 2   subject_id                      898 non-null    str    
 3   session_filename                898 non-null    str    
 4   session_id                      898 non-null    str    
 5   speaker_code                    898 non-null    str    
 6   speaker_role                    898 non-null    str    
 7   speaker_age_months              214 non-null    float64
 8   mlu_score                       898 non-null    float64
 9   mluw_score                      898 non-null    float64
 10  mlum_score                      898 non-null    float64
 11  ttr_score                       898 non-null    float64
 12  ipsyn_score                     898 non-null   

None

,language_id,experiment_name,subject_id,session_filename,session_id,speaker_code,speaker_role,speaker_age_months,mlu_score,mluw_score,mlum_score,ttr_score,ipsyn_score,n_utterances,text_part_of_speech,text_morphological_info,text_grammatical_relation_type,text_grammatical_dependency,text_phonological_info,text_raw,text_morphological_info_full,text_grammatical_relation_full,context_previous_utterance,context_next_utterance,meta_comments,meta_date,meta_languages,meta_location,meta_media,meta_number,meta_options,meta_other,meta_participants,meta_pid,meta_recording_quality,meta_room_layout,meta_situation,meta_tape_location,meta_time_duration,meta_time_start,meta_transcriber,meta_transcription,meta_types,meta_videos,meta_warning
0,Eng-NA,Brown,Eve,010600a.cha,010600a,CHI,Target_Child,18.0,1.430000,1.500000,1.430000,0.245714,14,741,"['adj noun', 'adj noun', 'adj noun', 'propn', ...","['more-Cmp-S1 cookie', 'more-Cmp-S1 cookie', '...","['AMOD ROOT PUNCT', 'AMOD ROOT PUNCT', 'AMOD R...","['1|2 2|0 3|2', '1|2 2|0 3|2', '1|2 2|0 3|2', ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['xxx more cookie .', 'xxx more cookie .', 'mo...","['adj|more-Cmp-S1 noun|cookie .', 'adj|more-Cm...","['1|2|AMOD 2|0|ROOT 3|2|PUNCT', '1|2|AMOD 2|0|...","[None, 'here you go .', 'you have another cook...","['you more cookies ?', 'you have another cooki...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
1,Eng-NA,Brown,Eve,010600a.cha,010600a,MOT,Mother,NaN,4.330000,4.320000,4.330000,0.371429,26,804,"['pron adj noun', 'intj det noun noun', 'aux p...","['you-Prs-Acc-S2 more-Cmp-S1 cookie-Plur', 'ho...","['NSUBJ AMOD ROOT PUNCT', 'DISCOURSE DET COMPO...","['1|3 2|3 3|0 4|3', '1|4 2|4 3|4 4|0 5|4', '1|...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['you 0v more cookies ?', 'how_about another g...",['pron|you-Prs-Acc-S2 adj|more-Cmp-S1 noun|coo...,"['1|3|NSUBJ 2|3|AMOD 3|0|ROOT 4|3|PUNCT', '1|4...","['more cookie .', 'you more cookies ?', 'how_a...","['how_about another graham cracker ?', 'would ...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
2,Eng-NA,Brown,Eve,010600a.cha,010600a,COL,Investigator,NaN,3.900000,3.900000,3.900000,0.504274,24,30,"['pron pron noun verb', 'aux pron verb pron', ...","['what-Int-S1 that-Dem your do-Part-Pres-S', '...","['ROOT NSUBJ NSUBJ ACL-RELCL PUNCT', 'AUX NSUB...","['1|0 2|4 3|4 4|1 5|1', '1|3 2|3 3|0 4|3 5|3',...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[""what's that you're doing ?"", 'can you do it ...",['pron|what-Int-S1 pron|that-Dem noun|your ver...,['1|0|ROOT 2|4|NSUBJ 3|4|NSUBJ 4|1|ACL-RELCL 5...,"['.', 'block broke .', 'can you do it ?', 'the...","['Mommy .', 'can you do it like that ?', 'ther...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
3,Eng-NA,Brown,Eve,010600a.cha,010600a,RIC,Investigator,NaN,3.615385,3.615385,3.615385,0.702128,20,13,"['pron aux pron verb', 'pron', 'pron adv verb ...",['what-Int-S1 be-Fin-Ind-Pres-S2 you-Prs-Nom-S...,"['OBJ AUX NSUBJ ROOT PUNCT', 'ROOT PUNCT', 'NS...","['1|4 2|4 3|4 4|0 5|4', '1|0 2|1', '1|3 2|3 3|...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","['what are you doing ?', 'what ?', ""I don('t) ...",['pron|what-Int-S1 aux|be-Fin-Ind-Pres-S2 pron...,['1|4|OBJ 2|4|AUX 3|4|NSUBJ 4|0|ROOT 5|4|PUNCT...,"['give dolly her bottle .', 'oh Fraser hat .',...","['water .', 'oh Fraser hat .', 'what_about the...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN
4,Eng-NA,Brown,Eve,010600b.cha,010600b,MOT,Mother,NaN,3.920

# CHAT Extra Engineering Pipeline

n_unintelligible: Count of xxx/yyy/www tokens representing communicative intent that was physically or phonetically unclear.

n_omissions: Count of words prefixed with 0 (e.g., 0det), used to track telegraphic speech and grammatical "holes" in development.

n_errors: Count of [*] markers indicating semantic, morphological, or phonological errors, vital for measuring accuracy.

n_fragments: Count of &+ markers used to identify broken words or phonological attempts before a full word is formed.

n_fillers: Count of &- markers (e.g., &um, &uh) which serve as a gold-standard metric for speech planning and fluency.

n_pauses: Count of (.), (..), and timed pauses to quantify hesitation, processing time, and conversational turn-taking rhythm.

n_events: Total count of &= (simple) and [=! ] (scoped) paralinguistics like laughing, crying, or pointing, capturing non-verbal engagement.

n_repetitions: Count of [/] markers where the child repeats words exactly, often used to study cognitive load or emphasis.

n_retracings: Count of [//] markers where a child stops and corrects themselves, a key indicator of emerging self-monitoring and syntax.

n_reformulations: Count of [///] markers representing total sentence restructuring, showing high-level linguistic flexibility.

n_interruptions: Count of +/. and +//. markers to identify how often the child is cut off or cuts themselves off mid-thought.

n_overlaps: Count of [<] and [>] markers to measure conversational timing and the child’s ability to navigate simultaneous speech.

n_imitations: Count of [+ IMIT] headers to quantify "parroting" behavior, essential for diagnosing developmental delays or structural priming.

n_responses: Count of [+ RES] headers used to distinguish between spontaneous utterances and those prompted by an adult.

n_emphatic: Count of [!!] markers to identify moments of high emotional salience or acoustic stress in the child's speech.

unint_ratio: The proportion of unintelligible tokens to total tokens, used to normalize "clarity" across sessions of different lengths.

imitation_score: A calculated bigram/unigram overlap between child and previous adult utterances to measure structural mimicry.

pos_sequence: A stringified sequence of Part-of-Speech tags used for stylometry and structural N-gram analysis without lexical bias.

morph_sequence: A stringified sequence of morphological features (e.g., PLUR, PRES) to track the acquisition of grammatical markers.

pos_variety_score: The count of unique POS tags used in a session, serving as a more robust measure of syntactic diversity than MLU alone.

text_clean: A "Deep Clean" NLP track where all metadata, replacements, and noise are stripped for use in Transformer models and Embeddings.

text_behavioral: A sequence modeling track where CHAT markers are replaced with explicit tokens (e.g., [UNINTEL]) to preserve structural "messiness."

env_adult_mlu: The average MLU of adults in the session, used to assess the linguistic complexity of the child's environment.

env_adult_pos_variety: The average syntactic variety of adults, measuring if the child is exposed to a rich or restricted vocabulary.

env_adult_text_clean: An aggregated text track of all adult speech in the session for environmental sentiment or topic modeling.

env_adult_pos_sequence: The structural sequence of adult speech to facilitate the study of cross-speaker structural priming.

env_adult_interruptions: Total count of adult-driven interruptions, used to quantify the "conversational pressure" placed on the child.

adult_to_child_utt_ratio: The ratio of adult utterances to child utterances, identifying if the session was adult-dominated or child-led.

In [64]:
# import pandas as pd
# import numpy as np
# import ast
# import re
# from typing import List, Dict, Any
# import logging

# logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
# logger = logging.getLogger(__name__)

# class CHILDESRefiner:
#     """
#     Expert-level refiner for CHILDES data. 
#     Transforms speaker-session Parquet files into Target_Child-centric units
#     with rigorous CHAT-manual parsing applied to all participants.
#     """
    
#     def __init__(self):
#         # Regex Patterns based strictly on the CHAT Manual specifications
#         self.patterns = {
#             # Replacements & Errors
#             'replacement': r'(?:\<[^>]+\>|\S+)\s+\[:\s+([^\]]+)\]', # [: target] 
#             'error': r'\[\*[^\]]*\]', # [* p], [* s], etc. [cite: 1487, 1934]
            
#             # Unidentifiable & Omitted
#             'unintelligible': r'\b(?:xxx|yyy|www)\b', # [cite: 902, 909, 913]
#             'omission': r'\b0[a-zA-Z_0-9]*\b', # 0det, 0v, or just 0 [cite: 952, 959, 1544]
            
#             # Fragments & Fillers
#             'fragment': r'&\+[a-zA-Z0-9_]+', # [cite: 919]
#             'filler': r'&-[a-zA-Z0-9_]+', # [cite: 926]
            
#             # Paralinguistics, Pauses, and Events
#             'simple_event': r'&=\S+', # &=laughs, &=points:car [cite: 1242, 1254]
#             'scoped_event': r'\[=!?[^\]]+\]|\[%[^\]]+\]', # [=! cries], [% com] [cite: 1376, 1404]
#             'pause': r'\(\.+|\(\d+(?::\d+)?\.\d*\)', # (.), (..), (1.5), (1:05.15) [cite: 1279, 1282, 1283]
            
#             # Disfluencies
#             'repetition': r'\[/\]', # [cite: 1437]
#             'retracing': r'\[//\]', # [cite: 1450]
#             'reformulation': r'\[///\]', # [cite: 1461]
#             'false_start': r'\[/-\]', # [cite: 1464]
#             'blocking': r'≠', # [cite: 1234]
            
#             # Interactional
#             'overlap': r'\[[><]\d*\]|\+<', # [<], [>], [>1], +< [cite: 1417, 1425, 1431]
#             'terminator_trailoff': r'\+\.\.\.\??', # +..., +..? [cite: 1292, 1301]
#             'terminator_interrupt': r'\+/\.\??', # +/., +/? [cite: 1304, 1313]
#             'terminator_self_interrupt': r'\+//\.\??' # +//., +//? [cite: 1314, 1321]
#         }

#     def _safe_parse(self, val: Any) -> List[str]:
#         """Safely evaluates stringified lists from Parquet."""
#         if pd.isna(val) or val == 'nan': return []
#         try:
#             return ast.literal_eval(val)
#         except:
#             return []

#     def _engineer_text_tracks(self, text_list: List[str]) -> Dict[str, Any]:
#         """
#         Processes CHAT syntax into Count Metrics, Behavioral Text, and Clean Text.
#         """
#         # Filter NaNs and join all utterances for the session into a single block
#         full_raw = " ".join([str(t) for t in text_list if pd.notna(t)])
        
#         # 1. Feature Engineering Matrix (Counts)
#         metrics = {
#             'n_unintelligible': len(re.findall(self.patterns['unintelligible'], full_raw)),
#             'n_omissions': len(re.findall(self.patterns['omission'], full_raw)),
#             'n_errors': len(re.findall(self.patterns['error'], full_raw)),
#             'n_fragments': len(re.findall(self.patterns['fragment'], full_raw)),
#             'n_fillers': len(re.findall(self.patterns['filler'], full_raw)),
#             'n_pauses': len(re.findall(self.patterns['pause'], full_raw)),
#             'n_events': len(re.findall(self.patterns['simple_event'], full_raw)) + len(re.findall(self.patterns['scoped_event'], full_raw)),
#             'n_repetitions': len(re.findall(self.patterns['repetition'], full_raw)),
#             'n_retracings': len(re.findall(self.patterns['retracing'], full_raw)),
#             'n_reformulations': len(re.findall(self.patterns['reformulation'], full_raw)),
#             'n_interruptions': len(re.findall(self.patterns['terminator_interrupt'], full_raw)),
#             'n_overlaps': len(re.findall(self.patterns['overlap'], full_raw))
#         }
        
#         # Calculate Unintelligibility Ratio
#         total_tokens = len(full_raw.split())
#         metrics['unint_ratio'] = metrics['n_unintelligible'] / total_tokens if total_tokens > 0 else 0

#         # 2. Track 1: text_clean (NLP-ready text)
#         cl_text = full_raw
#         # Resolve replacements: swap the error word with the target word [cite: 1392, 1396]
#         cl_text = re.sub(self.patterns['replacement'], r'\1', cl_text)
#         # Strip all CHILDES noise, codes, and events
#         for key in ['error', 'unintelligible', 'omission', 'fragment', 'filler', 
#                     'simple_event', 'scoped_event', 'pause', 'repetition', 'retracing', 
#                     'reformulation', 'false_start', 'blocking', 'overlap']:
#             cl_text = re.sub(self.patterns[key], '', cl_text)
#         # Normalize special terminators to standard punctuation
#         cl_text = re.sub(self.patterns['terminator_trailoff'], '...', cl_text)
#         cl_text = re.sub(self.patterns['terminator_interrupt'], '.', cl_text)
#         cl_text = re.sub(self.patterns['terminator_self_interrupt'], '.', cl_text)
#         cl_text = re.sub(r'[<>]', '', cl_text) # Remove scoping angle brackets 
#         metrics['text_clean'] = re.sub(r'\s+', ' ', cl_text).strip()

#         # 3. Track 2: text_behavioral (Sequence modeling tokens)
#         bh_text = full_raw
#         # Resolve replacements but flag them
#         bh_text = re.sub(self.patterns['replacement'], r'\1 [REPLACE]', bh_text)
#         # Substitute CHAT markers for explicit text tokens
#         substitutions = {
#             'error': ' [ERROR] ', 'unintelligible': ' [UNINTEL] ', 'omission': ' [OMISSION] ',
#             'fragment': ' [FRAG] ', 'filler': ' [FILLER] ', 'simple_event': ' [EVENT] ',
#             'scoped_event': ' [COMMENT] ', 'pause': ' [PAUSE] ', 'repetition': ' [REP] ',
#             'retracing': ' [RETR] ', 'reformulation': ' [REFORM] ', 'false_start': ' [FALSESTART] ',
#             'overlap': ' [OVERLAP] ', 'terminator_trailoff': ' [TRAILOFF] ', 
#             'terminator_interrupt': ' [INTERRUPT] ', 'terminator_self_interrupt': ' [SELF_INTERRUPT] '
#         }
#         for key, token in substitutions.items():
#             bh_text = re.sub(self.patterns[key], token, bh_text)
#         bh_text = re.sub(r'[<>]', '', bh_text)
#         metrics['text_behavioral'] = re.sub(r'\s+', ' ', bh_text).strip()

#         return metrics

#     def _calculate_imitation(self, child_utts: List[str], prev_utts: List[str]) -> float:
#         """Calculates bigram/unigram overlap to estimate imitation/echolalia."""
#         if not child_utts or not prev_utts: return 0.0
#         score = 0
#         for c, p in zip(child_utts, prev_utts):
#             if pd.isna(c) or pd.isna(p): continue
#             c_set = set(str(c).lower().split())
#             p_set = set(str(p).lower().split())
#             if not p_set: continue
#             score += len(c_set.intersection(p_set)) / len(p_set)
#         return score / len(child_utts)

#     def process(self, df: pd.DataFrame) -> pd.DataFrame:
#         df = df.copy()
#         if 'session_id' not in df.columns:
#             df = df.reset_index()

#         # Parse stringified lists
#         target_cols = ['text_raw', 'context_previous_utterance']
#         for col in target_cols:
#             if col in df.columns:
#                 df[col] = df[col].apply(self._safe_parse)

#         # Apply CHAT text extraction rules to ALL speakers (Child & Adults)
#         logger.info("Parsing CHAT semantics and engineering text tracks for all participants...")
#         chat_features = df['text_raw'].apply(self._engineer_text_tracks)
#         chat_df = pd.DataFrame(chat_features.tolist(), index=df.index)
#         df = pd.concat([df, chat_df], axis=1)

#         # Split into Target Child vs Environment (Adults/Peers)
#         child_df = df[df['speaker_role'].fillna('') == 'Target_Child'].copy()
#         other_df = df[df['speaker_role'].fillna('') != 'Target_Child'].copy()

#         # Calculate Imitation/Priming for the Child
#         child_df['imitation_score'] = child_df.apply(
#             lambda row: self._calculate_imitation(row['text_raw'], row['context_previous_utterance']), axis=1
#         )

#         # Collapse "Environment" (Adults) per session, aggregating their extracted CHAT metrics
#         logger.info("Aggregating Adult linguistic environment and behavioral metrics...")
#         env_agg = other_df.groupby('session_id').agg(
#             env_adult_mlu=('mlu_score', 'mean'),
#             env_adult_utterances=('n_utterances', 'sum'),
#             env_num_speakers=('speaker_code', 'nunique'),
#             env_roles_present=('speaker_role', lambda x: ", ".join(set(x))),
#             env_text_clean=('text_clean', lambda x: " ".join([str(t) for t in x if pd.notna(t)])),
#             env_text_behavioral=('text_behavioral', lambda x: " ".join([str(t) for t in x if pd.notna(t)])),
            
#             # Sum up the adults' behavioral frequencies to gauge environment complexity
#             env_adult_errors=('n_errors', 'sum'),
#             env_adult_pauses=('n_pauses', 'sum'),
#             env_adult_retracings=('n_retracings', 'sum'),
#             env_adult_interruptions=('n_interruptions', 'sum')
#         ).reset_index()

#         # Final Merge & Interaction Feature Calculation
#         final_df = pd.merge(child_df, env_agg, on='session_id', how='left')
        
#         final_df['adult_to_child_utt_ratio'] = (
#             final_df['env_adult_utterances'] / final_df['n_utterances']
#         ).replace([np.inf, -np.inf], 0).fillna(0)

#         # Final cleanup: drop messy raw lists and redundant columns
#         cols_to_drop = ['text_raw', 'context_previous_utterance', 'text_morphological_info_full', 'text_grammatical_relation_full']
#         final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns], inplace=True)
        
#         return final_df

import pandas as pd
import numpy as np
import ast
import re
from typing import List, Dict, Any
import logging

class CHILDESRefiner:
    def __init__(self):
        self.patterns = {
            # Aggressive cleaning patterns
            'all_brackets': r'\[.*?\]', # Strips [+ RES], [!!], [^ whispers], etc.
            'word_markers': r'@\w+',    # Strips @o, @l, @c
            'internal_symbols': r'[:·\(\)]', # Strips lengthening (:) and pauses in words
            'punctuation': r'[?!.]',
            'whitespace': r'\s+',
            
            # Feature extraction patterns (for the metrics)
            'imitation_flag': r'\[\+ IMIT\]',
            'response_flag': r'\[\+ RES\]',
            'emphasis_flag': r'\[!!\]',
            'replacement': r'(?:\<[^>]+\>|\S+)\s+\[:\s+([^\]]+)\]',
        }

    def _safe_parse(self, val: Any) -> List[str]:
        if isinstance(val, list): return val
        if pd.isna(val) or val == 'nan' or val == '': return []
        try:
            return ast.literal_eval(val)
        except:
            return []

    def _clean_string_strictly(self, text: str) -> str:
        """The 'Deep Clean' logic for standard NLP."""
        # 1. Resolve replacements first ([: target])
        text = re.sub(self.patterns['replacement'], r'\1', text)
        # 2. Strip all bracketed metadata
        text = re.sub(self.patterns['all_brackets'], '', text)
        # 3. Strip @markers (quack@o -> quack)
        text = re.sub(self.patterns['word_markers'], '', text)
        # 4. Strip internal lengthening/pauses (m:hm -> mhm)
        text = re.sub(self.patterns['internal_symbols'], '', text)
        # 5. Final whitespace cleanup
        text = re.sub(self.patterns['whitespace'], ' ', text).strip()
        return text

    def _engineer_structural_sequences(self, df: pd.DataFrame) -> pd.DataFrame:
        """Converts structural lists into flattened sequences for ML/NLP."""
        # POS Sequence (e.g., "NOUN VERB DET NOUN")
        df['pos_sequence'] = df['text_part_of_speech'].apply(
            lambda x: " ".join([str(i) for i in self._safe_parse(x)])
        )
        
        # Morphological Sequence (e.g., "PLUR PRES 3S")
        df['morph_sequence'] = df['text_morphological_info'].apply(
            lambda x: " ".join([str(i) for i in self._safe_parse(x)])
        )

        # Complexity Feature: POS Variety Score
        df['pos_variety_score'] = df['text_part_of_speech'].apply(
            lambda x: len(set(self._safe_parse(x)))
        )
        
        return df

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        # 1. Structural Sequences (Do this before dropping columns)
        df = self._engineer_structural_sequences(df)

        # 2. Parsing text_raw into Clean and Behavioral
        def process_text_row(text_list):
            raw_joined = " ".join([str(t) for t in self._safe_parse(text_list)])
            
            # Extraction for Numeric Features
            metrics = {
                'n_imitations': len(re.findall(self.patterns['imitation_flag'], raw_joined)),
                'n_responses': len(re.findall(self.patterns['response_flag'], raw_joined)),
                'n_emphatic': len(re.findall(self.patterns['emphasis_flag'], raw_joined)),
                'text_clean': self._clean_string_strictly(raw_joined),
                # Behavioral track keeps markers but cleans punctuation
                'text_behavioral': re.sub(self.patterns['whitespace'], ' ', raw_joined).strip()
            }
            return metrics

        text_results = df['text_raw'].apply(process_text_row)
        text_df = pd.DataFrame(text_results.tolist(), index=df.index)
        df = pd.concat([df, text_df], axis=1)

        # 3. Separate Target Child and Environment
        child_df = df[df['speaker_role'] == 'Target_Child'].copy()
        other_df = df[df['speaker_role'] != 'Target_Child'].copy()

        # 4. Aggregate Environment (Adults)
        # We now include the new structural sequences in the adult environment
        env_agg = other_df.groupby('session_id').agg(
            env_adult_mlu=('mlu_score', 'mean'),
            env_adult_pos_variety=('pos_variety_score', 'mean'),
            env_adult_text_clean=('text_clean', lambda x: " ".join(x)),
            env_adult_pos_sequence=('pos_sequence', lambda x: " ".join(x)),
            env_total_imitations=('n_imitations', 'sum')
        ).reset_index()

        # 5. Final Merge
        final_df = pd.merge(child_df, env_agg, on='session_id', how='left')
        
        # Drop raw list columns to keep it "Clean"
        cols_to_drop = ['text_raw', 'text_part_of_speech', 'text_morphological_info', 
                        'text_grammatical_relation_type', 'text_grammatical_dependency',
                        'text_phonological_info']
        final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns], inplace=True)

        return final_df

In [65]:
# Example call:
file_path = '../data/processed/Eng-NA/Brown/_Brown_extracted.parquet'
raw_df = pd.read_parquet(file_path)

refiner = CHILDESRefiner()
processed_df = refiner.process(raw_df)

In [66]:
display(processed_df.info(), processed_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 52 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   language_id                     214 non-null    str    
 1   experiment_name                 214 non-null    str    
 2   subject_id                      214 non-null    str    
 3   session_filename                214 non-null    str    
 4   session_id                      214 non-null    str    
 5   speaker_code                    214 non-null    str    
 6   speaker_role                    214 non-null    str    
 7   speaker_age_months              214 non-null    float64
 8   mlu_score                       214 non-null    float64
 9   mluw_score                      214 non-null    float64
 10  mlum_score                      214 non-null    float64
 11  ttr_score                       214 non-null    float64
 12  ipsyn_score                     214 non-null   

None

,language_id,experiment_name,subject_id,session_filename,session_id,speaker_code,speaker_role,speaker_age_months,mlu_score,mluw_score,mlum_score,ttr_score,ipsyn_score,n_utterances,text_morphological_info_full,text_grammatical_relation_full,context_previous_utterance,context_next_utterance,meta_comments,meta_date,meta_languages,meta_location,meta_media,meta_number,meta_options,meta_other,meta_participants,meta_pid,meta_recording_quality,meta_room_layout,meta_situation,meta_tape_location,meta_time_duration,meta_time_start,meta_transcriber,meta_transcription,meta_types,meta_videos,meta_warning,pos_sequence,morph_sequence,pos_variety_score,n_imitations,n_responses,n_emphatic,text_clean,text_behavioral,env_adult_mlu,env_adult_pos_variety,env_adult_text_clean,env_adult_pos_sequence,env_total_imitations
0,Eng-NA,Brown,Eve,010600a.cha,010600a,CHI,Target_Child,18.0,1.43,1.50,1.43,0.245714,14,741,"['adj|more-Cmp-S1 noun|cookie .', 'adj|more-Cm...","['1|2|AMOD 2|0|ROOT 3|2|PUNCT', '1|2|AMOD 2|0|...","[None, 'here you go .', 'you have another cook...","['you more cookies ?', 'you have another cooki...",NaN,15-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034743-1,NaN,NaN,NaN,NaN,10:00-11:00,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN,,,0,35,79,2,xxx more cookie . xxx more cookie . more juice...,xxx more cookie . xxx more cookie . more juice...,3.948462,12.333333,you 0v more cookies ? how_about another graham...,pron pron noun verb aux pron verb pron aux pr...,1
1,Eng-NA,Brown,Eve,010600b.cha,010600b,CHI,Target_Child,18.0,1.80,1.91,1.80,0.314286,18,473,"['num|one num|two num|three .', 'num|one num|t...","['1|3|NUMMOD 2|3|NUMMOD 3|0|ROOT 4|3|PUNCT', '...","['one two three four .', 'one two three .', ""i...","['one two three two one two three two .', 'no ...",NaN,29-OCT-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='MOT', name='', role='Mother...",11312/c-00034744-1,NaN,NaN,NaN,NaN,10:30-11:30,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN,,,0,14,60,0,one two three . one . two . three . two . one ...,one two three . [+ IMIT] one (.) two (.) three...,3.942237,0.000000,one two three four . no four . it's one . two ...,,0
2,Eng-NA,Brown,Eve,010700a.cha,010700a,CHI,Target_Child,19.0,2.14,2.16,2.14,0.342857,16,253,"['adj|more-Cmp-S1 noun|coffee .', 'adj|more-Cm...","['1|2|AMOD 2|0|ROOT 3|2|PUNCT', '1|3|AMOD 2|3|...","[None, 'more coffee ?', 'Eve !', 'no .', ""yes ...","['more coffee ?', ""that's better ."", ""that's ...",NaN,12-NOV-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', name='', role='Target...",11312/c-00034745-1,NaN,NaN,NaN,NaN,10:30-11:30,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN,,,0,13,37,0,more coffee . more grape juice too . no . oh F...,more coffee . more grape juice too . [+ RES] n...,4.023651,21.000000,"more coffee ? that's better . Eve , don't play...",adp noun adv intj cm pron verb pron aux adp n...,2
3,Eng-NA,Brown,Eve,010700b.cha,010700b,CHI,Target_Child,19.0,2.07,2.07,2.07,0.317143,18,585,"['o|peep .', 'intj|yeah .', 'noun|block-Plur ....","['1|0|ROOT 2|1|PUNCT', '1|0|ROOT 2|1|PUNCT', '...","['put the stool back , please .', 'can you get...","['can you get the blocks out ?', 'blocks .', '...",NaN,26-NOV-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='MOT', name='', role='Mother...",11312/c-00034746-1,NaN,NaN,NaN,NaN,10:30-11:45,NaN,NaN,NaN,"long, toyplay, TD",NaN,NaN,,,0,35,93,4,peep peep . yeah . blocks . one . one . two . ...,peep@o [/] peep@o . yeah . [+ RES] blocks . on...,2.816667,159.333333,xxx you put the stool back and I'll open the t...,pron verb det noun adv cconj adv verb det noun...,0
4,Eng-NA,Brown,Eve,010800.cha,010800,CHI,Target_Child,20.0,2.16,2.19,2.16,0.294286,24,706,"['adj|more-Cmp-S1 noun|grape noun|juice .', 'i...","['1|3|AMOD 2|3|COMPOUND 3|0|ROOT 4|3|PUNCT', '...","[None, 'more ?', ""didn't think so ."", 'put the...","['more ?', ""didn't think so ."", 'you must put ...",NaN,10-DEC-1962,['eng'],NaN,NaN,NaN,NaN,NaN,"[Participant(code='CHI', n

In [67]:
processed_df.text_clean[0]

'xxx more cookie . xxx more cookie . more juice ? Fraser . Fraser . Fraser . Fraser . yeah . xxx that ? a fly . fly . Mommy telephone . my telephone . xxx . Mommy . no . man man . xxx . xxx more cookie . block broke . there . I did it . there . there . Fraser . xxx . xxx . baby . Mommy read . a stool . Fraser . Fraser . xxx more cookie . xxx more cookie . <little little little> little . milk ? <milk milk> milk . that ? Fraser water ? oh Fraser . bye . +< xxx water . Fraser water . Fraser water . that ? coffee . Fraser coffee . down down . cookie . Mommy . Mommy . a fly . read the puzzle . read the puzzle . read the puzzle . read the puzzle . <read the puzzle read the puzzle> read the puzzle . Racketyboom . hat . xxx . mhm . +< xxx water . bottle . +< xxx water . there . Fraser hat . oh Fraser hat . <oh Fraser hat> oh Fraser hat . Fraser hat . no . eye . that . xxx . xxx . that . soldier . soldier . that that . Fraser hat . Eve find it . that that . man . Eve . +< xxx xxx down . that . 

In [69]:
processed_df.pos_sequence.unique()

<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    